# Practice 19 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: DateTime + MultiIndex
A flight delays DataFrame has been created for you.

1. Convert `flight_date` to datetime. Extract `year` and `month`.
2. Add `days_ago` = days between `flight_date` and `2026-03-17`. Find mean and median with NumPy.
3. Set a `(airline, route)` MultiIndex. Sort it.
4. Retrieve all rows for `'Delta'`.
5. Use `pd.IndexSlice` to get all `'JFK-LAX'` rows across every airline.

In [4]:
# Starter data — don't change this
flights = pd.DataFrame({
    'airline':     ['Delta', 'Delta', 'United', 'United', 'Southwest', 'Southwest', 'Delta', 'United'],
    'route':       ['JFK-LAX', 'ORD-MIA', 'JFK-LAX', 'DEN-SEA', 'ORD-MIA', 'JFK-LAX', 'DEN-SEA', 'ORD-MIA'],
    'flight_date': ['2024-03-10', '2024-06-22', '2024-08-05', '2024-10-14',
                    '2024-12-01', '2025-02-18', '2025-05-07', '2025-08-30'],
    'delay_min':   [12, 45, 0, 78, 23, 5, 102, 31],
    'passengers':  [182, 210, 165, 190, 143, 178, 201, 156],
})

# Your code here
flights['flight_date'] = pd.to_datetime(flights['flight_date'])
flights['flight_date'].dt.year
flights['flight_date'].dt.month
flights['days_ago'] = (pd.to_datetime('2026-03-17')-flights['flight_date']).dt.days
np.mean(flights['days_ago'])
np.median(flights['days_ago'])
flights = flights.set_index(['airline','route']).sort_index()
flights.loc['Delta']
idx = pd.IndexSlice
flights.loc[idx[:,'JFK-LAX'],:]


,,flight_date,delay_min,passengers,days_ago
airline,route,,,,
Delta,JFK-LAX,2024-03-10,12,182,737
Southwest,JFK-LAX,2025-02-18,5,178,392
United,JFK-LAX,2024-08-05,0,165,589


---
## Level 2 — Transformations

### Task 2: stack() / unstack() / .xs()
A wide DataFrame of monthly website traffic (thousands of visits) by country has been created for you.

1. Stack to long format. Store as `traffic_long` (use `.to_frame('visits')`)
2. Add `log_visits` using NumPy
3. Add `high_traffic`: `True` if `visits` > 150 (use `np.where`)
4. Use `.xs()` to extract all rows for `'Mar'` across every country
5. Unstack back to wide format
6. Find the country with the highest mean traffic using NumPy

In [ ]:
# Starter data — don't change this
np.random.seed(9)
web = pd.DataFrame({
    'country': ['Brazil', 'Germany', 'Japan', 'Nigeria'],
    'Jan':     np.random.randint(60, 300, size=4),
    'Feb':     np.random.randint(60, 300, size=4),
    'Mar':     np.random.randint(60, 300, size=4),
    'Apr':     np.random.randint(60, 300, size=4),
}).set_index('country')

# Your code here
traffic_long = web.stack().to_frame('visits')
traffic_long['log_visits'] = np.log(traffic_long['visits'])
traffic_long['high_traffic'] = np.where(traffic_long['visits']>150, True, False)
traffic_long
#traffic_long.xs('Mar',level=)
w = traffic_long.unstack()
cm = traffic_long.groupby(level='country')['visits'].mean()
cm.idxmax()



TypeError: 'NoneType' object is not callable

### Task 3: .apply()
A food delivery orders DataFrame has been created for you.

1. Add `order_total`: row-wise `item_price * quantity` using `apply` with `axis=1`
2. Add `price_tier`: apply a lambda to `item_price` — `'Premium'` if > 20, else `'Standard'`
3. Add `priority` (row-wise): `'High'` if `order_total > 80` **and** `distance_km < 5`, `'Medium'` if either is true, else `'Low'`

In [21]:
# Starter data — don't change this
np.random.seed(31)
deliveries = pd.DataFrame({
    'order_id':    [f'ORD{i:03d}' for i in range(1, 9)],
    'item':        ['Burger', 'Sushi', 'Pizza', 'Salad', 'Tacos', 'Ramen', 'Steak', 'Pasta'],
    'item_price':  [12.5, 28.0, 18.0, 9.5, 14.0, 22.0, 35.0, 16.5],
    'quantity':    np.random.randint(1, 6, size=8),
    'distance_km': np.round(np.random.uniform(1.0, 12.0, size=8), 1),
})

# Your code here

deliveries['order_total'] = deliveries.apply(lambda row: row['item_price']* row['quantity'], axis = 1)
deliveries['price_tier'] = deliveries['item_price'].apply(lambda x: 'Premium' if x>20 else 'Standard')
deliveries
deliveries['priority']= deliveries.apply(
    lambda row: 'High' if row['order_total']>80 and row['distance_km']<5
    else 'Medium' if row['order_total']>80 or row['distance_km']<5
    else 'Low', axis = 1)

### Task 4: .str + .rank()
A job listings DataFrame has been created for you.

1. Add `title_upper`: job titles in uppercase
2. Filter to listings where `title` contains `'Engineer'` (case-insensitive)
3. Extract the numeric part of `job_id` (e.g. `'J04'` → `'04'`) using `.str` slicing. Store as `num`.
4. Add `salary_rank`: rank by `salary`, highest first
5. Add `dept_rank`: rank within each `dept` by `salary`, highest first
6. Filter to the top-ranked listing per dept (`dept_rank == 1`)

In [27]:
# Starter data — don't change this
np.random.seed(52)
jobs = pd.DataFrame({
    'job_id':   [f'J{i:02d}' for i in range(1, 9)],
    'title':    ['Software Engineer', 'Data Analyst', 'ML Engineer', 'Product Manager',
                 'DevOps Engineer', 'UX Designer', 'Data Engineer', 'Scrum Master'],
    'dept':     ['Engineering', 'Analytics', 'Engineering', 'Product',
                 'Engineering', 'Design', 'Analytics', 'Product'],
    'salary':   np.random.randint(70, 180, size=8) * 1000,
    'openings': np.random.randint(1, 6, size=8),
})

# Your code here
jobs['title_upper'] = jobs['title'].str.upper()
jobs[jobs['title'].str.contains('Engineer')]
jobs['num'] = jobs['job_id'].str[1:]
jobs['salary_rank'] = jobs['salary'].rank(ascending=False)
jobs['dept_rank'] = jobs.groupby('dept')['salary'].rank(ascending=False)
jobs[jobs['dept_rank'] == 1]

,job_id,title,dept,salary,openings,title_upper,num,salary_rank,dept_rank
4,J05,DevOps Engineer,Engineering,156000,4,DEVOPS ENGINEER,05,1.0,1.0
5,J06,UX Designer,Design,98000,4,UX DESIGNER,06,4.5,1.0
6,J07,Data Engineer,Analytics,102000,2,DATA ENGINEER,07,3.0,1.0
7,J08,Scrum Master,Product,139000,4,SCRUM MASTER,08,2.0,1.0


---
## Level 3 — Aggregation

### Task 5: pd.merge() + groupby + transform
Two DataFrames have been created for you: `stores` and `transactions`.

1. Inner join on `store_id`
2. Add `region_avg_amount`: mean transaction `amount` per `region` using `transform`
3. Add `above_region_avg`: `True` if `amount` > `region_avg_amount`
4. Total `amount` per `region` — sort descending
5. Total `amount` per `store_type` using groupby

In [32]:
# Starter data — don't change this
np.random.seed(64)
stores = pd.DataFrame({
    'store_id':   [f'ST{i:02d}' for i in range(1, 9)],
    'region':     ['North', 'North', 'South', 'South', 'East', 'East', 'West', 'West'],
    'store_type': ['Mall', 'Standalone', 'Mall', 'Standalone', 'Mall', 'Standalone', 'Mall', 'Standalone'],
})

transactions = pd.DataFrame({
    'txn_id':   [f'T{i:03d}' for i in range(1, 25)],
    'store_id': np.random.choice([f'ST{i:02d}' for i in range(1, 9)], size=24),
    'amount':   np.random.randint(20, 500, size=24),
    'month':    np.random.choice(['Jan', 'Feb', 'Mar'], size=24),
})

# Your code here
ij = pd.merge(
    stores,
    transactions,
    how="inner",
    on="store_id"
)
ij['region_avg_amount'] = ij.groupby('region')['amount'].transform('mean')
ij['above_region_avg'] = ij['amount']>ij['region_avg_amount']
ij.groupby('region')['amount'].sum().sort_values(ascending=False)
ij.groupby('store_type')['amount'].sum().sort_values(ascending=False)



store_type
Mall          4650
Standalone    1600
Name: amount, dtype: int64

### Task 6: pd.concat() + pivot_table + stack/unstack
Two quarterly revenue DataFrames have been created for you.

1. Concatenate `rev_h1` and `rev_h2` vertically. Reset the index. Store as `revenue`.
2. Pivot: total `revenue` by `product` (rows) and `quarter` (columns) — use `aggfunc='sum'`
3. Stack the pivot result to long format
4. Use `.xs()` to pull out just `'Q2'` from the stacked result
5. Which product had the highest total revenue? Sum across quarters on the pivot, then use `.idxmax()`.

In [43]:
# Starter data — don't change this
np.random.seed(71)
rev_h1 = pd.DataFrame({
    'product': np.random.choice(['Widget', 'Gadget', 'Gizmo'], size=9),
    'region':  np.random.choice(['APAC', 'EMEA', 'Americas'], size=9),
    'revenue': np.random.randint(1000, 8000, size=9),
    'quarter': np.random.choice(['Q1', 'Q2'], size=9),
})

rev_h2 = pd.DataFrame({
    'product': np.random.choice(['Widget', 'Gadget', 'Gizmo'], size=9),
    'region':  np.random.choice(['APAC', 'EMEA', 'Americas'], size=9),
    'revenue': np.random.randint(1000, 8000, size=9),
    'quarter': np.random.choice(['Q3', 'Q4'], size=9),
})

# Your code here
revenue = pd.concat(
    [rev_h1, rev_h2],
    ignore_index=True
)
p = revenue.pivot_table(
    index='product',
    columns='quarter',
    values='revenue',
    aggfunc='sum'
)
ps = p.stack().to_frame('revenue')
ps
ps.xs('Q2',level='quarter')
ps
sm = ps.groupby(level='product')['revenue'].sum()
sm.idxmax()

'Gizmo'

---
## Level 4 — Real-world

### Task 7: Full Pipeline
Three DataFrames have been created for you: `researchers`, `experiments`, and `results`.

1. Convert `start_date` to datetime. Add `days_running` = days from `start_date` to `2026-03-17`. Find mean and median with NumPy.
2. Inner join `results` with `researchers` on `researcher_id`. Inner join with `experiments` on `experiment_id`. Drop nulls.
3. Use `.apply()` to add `outcome` (row-wise): `'Success'` if `score > 80` **and** `trials > 5`, `'Partial'` if either is true, else `'Fail'`
4. Add `lab_avg_score`: mean `score` per `lab` using `transform`
5. Pivot: mean `score` by `lab` (rows) and `field` (columns) using `pivot_table`. Stack the result to long format.
6. Use `.xs()` on the stacked result to pull out just `'Biology'`
7. Bin `score` into 3 equal-width groups labelled `['Low', 'Medium', 'High']`. Count rows per group (use `observed=True`).
8. Find the correlation between `score` and `trials` using NumPy

In [54]:
# Starter data — don't change this
np.random.seed(48)
n = 10

researchers = pd.DataFrame({
    'researcher_id': [f'R{i:02d}' for i in range(1, n + 1)],
    'name':          [f'Dr. {chr(65+i)}' for i in range(n)],
    'lab':           np.random.choice(['Alpha', 'Beta', 'Gamma'], size=n),
    'start_date':    pd.date_range('2022-01-01', periods=n, freq='75D').strftime('%Y-%m-%d'),
})

experiments = pd.DataFrame({
    'experiment_id': [f'EX{i:02d}' for i in range(1, 7)],
    'field':         ['Biology', 'Chemistry', 'Biology', 'Physics', 'Chemistry', 'Physics'],
    'budget':        [15000, 22000, 18000, 30000, 12000, 25000],
})

results = pd.DataFrame({
    'result_id':     [f'RS{i:02d}' for i in range(1, 25)],
    'researcher_id': np.random.choice([f'R{i:02d}' for i in range(1, n + 1)], size=24),
    'experiment_id': np.random.choice([f'EX{i:02d}' for i in range(1, 7)], size=24),
    'score':         np.random.randint(40, 100, size=24),
    'trials':        np.random.randint(1, 12, size=24),
})

# Your code here


researchers['start_date'] = pd.to_datetime(researchers['start_date'])
researchers['days_running'] = (pd.to_datetime('2026-03-17')-researchers['start_date']).dt.days
np.mean(researchers['days_running'])
np.median(researchers['days_running'])
ij = pd.merge(
    researchers,
    results ,
    on='researcher_id',
    how='inner'
)
ij2 = pd.merge(
    ij,
    experiments,
    how='inner',
    on='experiment_id'
).dropna()
ij2['outcome'] = ij2.apply(lambda row: 'Success' if row['score']>80 and row['trials']>5
                           else 'Partial' if row['score']>80 or row['trials']>5
                           else 'Fail', axis = 1)
ij2['lab_avg_score'] = ij2.groupby('lab')['score'].transform('mean')
p = ij2.pivot_table(
    index = 'lab',
    columns = 'field',
    values = 'score'   
).stack().to_frame('score')
p.xs('Biology',level='field')
ij2['sg'] = pd.cut(ij2['score'],3,labels=['Low','Medium','High'])
ij2.groupby('sg', observed=True).agg('count')
np.corrcoef(ij2['score'], ij2['trials'])

array([[1.        , 0.06965433],
       [0.06965433, 1.        ]])